# Adoption Persona Builder: a multi-agent pipeline (Claude API)

This notebook turns interview transcripts into **adoption personas**. Five groups of agents run one after another, and each group builds on what the previous one produced:

| # | Agent group | Agents inside | Output |
|---|---|---|---|
| 1 | **Familiarization & speaker identification** | one *Reader* per transcript (in parallel) + a *Familiarization Lead* | who is the interviewer and who is the participant, each participant's profile and **user group**, a dataset overview |
| 2 | **Thematic analysis** (Braun & Clarke) | one *Coder* per transcript (in parallel) + a *Theme Lead* + a *Theme Reviewer* | codes with quotes, then themes, each **flagged** as shared, group-specific or divergent across user groups |
| 3 | **Theory-informed user attributes** | three *Theory Lens* agents per participant: **Diffusion of Innovation**, **TAM / UTAUT**, **UX principles** + an *Attribute Synthesizer* | a 1–5 score with evidence for every adoption attribute, per participant and per user group |
| 4 | **Cluster analysis** | statistical clustering in Python (Ward hierarchical, silhouette to choose *k*) + a *Cluster Analyst* agent | adoption segments, each mapped to Rogers' adopter categories |
| 5 | **Persona cards** | one *Persona Designer* per cluster (in parallel); quotes are checked against the transcripts | HTML persona cards you can download |

### How to run it in Google Colab
1. Click the 🔑 **Secrets** icon in the left sidebar. Add `ANTHROPIC_API_KEY` with your key and turn on **Notebook access**.
2. **Runtime → Run all.** The first run uses the six built-in sample transcripts so you can see the whole flow.
3. In **Step 3**, set `USE_SAMPLE_DATA = False` and upload your own transcripts (`.txt`, `.docx`, `.vtt`, `.srt`), one file per interview.

> Clustering needs enough people to be meaningful. For real studies, aim for at least about 12 participants. With 6 samples the clusters are only a demo.

## Step 1: Install

In [ ]:
!pip install -q anthropic python-docx scikit-learn

## Step 2: Settings and connection to Claude

In [ ]:
import anthropic, json, os, re
from concurrent.futures import ThreadPoolExecutor

try:
    from google.colab import userdata
    API_KEY = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    API_KEY = os.environ["ANTHROPIC_API_KEY"]

client = anthropic.Anthropic(api_key=API_KEY)

MODEL = "claude-opus-5-5"   # use "claude-sonnet-5-5" for a cheaper run
MAX_PARALLEL = 4            # how many agents run at the same time (lower it if you hit rate limits)
OUTPUT_DIR = "persona_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESEARCH_CONTEXT = """
Study: adoption of an AI-powered literature-review assistant at a university.
Goal: understand what drives or blocks adoption, and build adoption personas
to guide design and rollout.
"""

# The user groups in your study. Leave the list empty to let the agents detect groups.
USER_GROUPS = ["Faculty", "PhD students"]


def run_agent(role: str, task: str, output_model=None, effort: str = "medium"):
    """Run one agent. role = its system prompt. If output_model (a Pydantic class) is
    given, the answer comes back as a validated object instead of text."""
    kwargs = dict(
        model=MODEL,
        max_tokens=64000,
        system=role,
        messages=[{"role": "user", "content": task}],
        output_config={"effort": effort},           # low | medium | high | xhigh | max
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",                         # if Claude declines, retry on a fallback model
    )
    if output_model is not None:
        kwargs["output_format"] = output_model
    with client.beta.messages.stream(**kwargs) as stream:
        msg = stream.get_final_message()

    if msg.stop_reason == "refusal":
        raise RuntimeError(f"Request declined: {msg.stop_details}")
    if msg.stop_reason == "max_tokens":
        raise RuntimeError("Answer was cut off (max_tokens). Shorten the input or raise max_tokens.")
    if output_model is not None:
        return msg.parsed_output
    return "".join(b.text for b in msg.content if b.type == "text")


def parallel(fn, items):
    with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as pool:
        return list(pool.map(fn, items))


def save_json(name, obj):
    with open(os.path.join(OUTPUT_DIR, name), "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

print(run_agent("You are a helpful assistant.", "Reply with the single word: ready", effort="low"))

## Step 3: Load the transcripts

Use one file per interview. Speaker labels can be anything (`Speaker 1:`, `A:`, names, timestamps). Group 1 works out who is who.

In [ ]:
USE_SAMPLE_DATA = True

SAMPLE_TRANSCRIPTS = {
"interview_01.txt": """Speaker 1: Thanks for joining. Could you tell me about your role?
Speaker 2: Sure. I'm an associate professor in computer science, about twelve years at the university. I supervise six PhD students.
Speaker 1: Have you used the AI literature-review assistant?
Speaker 2: Yes, from the first week of the pilot. Honestly, I sign up for every new tool. I like being the first to figure out what works.
Speaker 1: What value do you get from it?
Speaker 2: It cuts my scoping time in half. I paste a research question and get a map of the field in minutes. It fits how I already work in Zotero.
Speaker 1: Any concerns?
Speaker 2: I always check the citations, sometimes it invents a reference. But I can see the sources, so I trust it enough.
Speaker 1: Do colleagues use it?
Speaker 2: Not many. I showed it in a department meeting and a few got curious once they saw my results.""",

"interview_02.txt": """Speaker A: Before we start, I should say I am not a fan of these AI things.
Speaker B: That's completely fine, we want all views. What is your role?
Speaker A: Professor of history. Thirty years of teaching. I read sources myself, that is the job.
Speaker B: Have you tried the assistant?
Speaker A: Once, because the dean asked. It gave me a summary of an archive I know well, and it was wrong in two places. Why would I trust it?
Speaker B: What would make you consider it again?
Speaker A: If it showed me exactly where every claim comes from, page numbers. And if it did not take hours to learn. I do not have time for training sessions.
Speaker B: Does your department talk about it?
Speaker A: The younger staff do. I am happy with my methods; they have worked for decades.""",

"interview_03.txt": """Interviewer: Can you introduce yourself?
Lina: I'm a second-year PhD student in public health, 26 years old.
Interviewer: How do you use the assistant?
Lina: Every day! It's so easy, I just ask questions in plain language. It found papers my keyword searches missed.
Interviewer: How did you start?
Lina: A friend in my lab showed me, and I tried the free version for a weekend before the university licence. Being able to play with it first was key.
Interviewer: Any worries?
Lina: A bit about privacy, I don't upload unpublished data. But the benefits are huge, I feel faster and less overwhelmed.
Interviewer: What about your supervisor?
Lina: She's neutral, she just wants me to verify everything.""",

"interview_04.txt": """[00:00:05] Speaker 1: I'm a third-year PhD student in economics.
[00:00:12] Speaker 2: And how familiar are you with the AI assistant?
[00:00:15] Speaker 1: I use it, but carefully. My supervisor is very strict about academic integrity, so I worry she'll think I'm cheating.
[00:00:31] Speaker 2: What do you use it for?
[00:00:34] Speaker 1: Finding papers and checking if I missed a key study. Not for writing. It's useful, clearly, it saves me time.
[00:00:52] Speaker 2: What would help you use it more?
[00:00:55] Speaker 1: Clear university guidelines on what is allowed. If the graduate school said it's fine, I'd use it a lot more. And the interface is OK but there are too many settings.
[00:01:20] Speaker 2: Do other students use it?
[00:01:23] Speaker 1: Most of my cohort does, quietly.""",

"interview_05.txt": """Speaker 2: Please tell me about yourself and your work.
Speaker 1: I'm an assistant professor in nursing, about four years here. I teach a lot and do clinical research.
Speaker 2: Have you adopted the assistant?
Speaker 1: Partly. I see the potential, but I need IT to set it up with our library database, and that has not happened. Without access to the full texts it's half useful.
Speaker 2: What else matters for you?
Speaker 1: Time. If there were a short workshop and a help desk, I'd move my whole review workflow to it. I'm waiting to see how others in my faculty find it.
Speaker 2: Any concerns about accuracy?
Speaker 1: Some, but if it shows sources I can verify like any research assistant.""",

"interview_06.txt": """Speaker 1: Hi, can you tell me a little about yourself?
Speaker 2: I'm a first-year PhD student in education. I came back to study after ten years teaching in schools.
Speaker 1: Have you used the AI assistant?
Speaker 2: I tried, but I got lost. I'm not very good with technology, and I didn't know what to type. I felt stupid.
Speaker 1: What would help?
Speaker 2: Step-by-step examples. Someone sitting with me the first time. Maybe templates for prompts.
Speaker 1: Do you think it would be useful once you learn it?
Speaker 2: Yes, my classmates say it saves them hours. I just need confidence first. And I'm scared of making mistakes that my supervisor would notice.""",
}

def read_file(name: str, data: bytes) -> str:
    if name.lower().endswith(".docx"):
        import docx, io
        return "\n".join(p.text for p in docx.Document(io.BytesIO(data)).paragraphs)
    return data.decode("utf-8", errors="ignore")

if USE_SAMPLE_DATA:
    raw_files = SAMPLE_TRANSCRIPTS
else:
    from google.colab import files
    uploaded = files.upload()
    raw_files = {name: read_file(name, data) for name, data in uploaded.items()}

transcripts = [
    {"participant_id": f"P{i:02d}", "file": name, "raw_text": text}
    for i, (name, text) in enumerate(sorted(raw_files.items()), start=1)
]
for t in transcripts:
    print(t["participant_id"], t["file"], f"{len(t['raw_text']):,} chars")

## Step 4: Output schemas

Every agent returns structured data, which keeps the hand-offs between groups reliable.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional

# ---- Group 1: familiarization & speakers ----
class SpeakerRole(BaseModel):
    speaker_label: str = Field(description="The label exactly as written in the transcript, e.g. 'Speaker 1', 'A', 'Lina'")
    role: Literal["interviewer", "participant", "other"]
    reasoning: str

class ParticipantProfile(BaseModel):
    user_group: str = Field(description="Which user group the participant belongs to")
    role_or_occupation: str
    demographics: str = Field(description="Only what is stated: age, years of experience, field, etc. Write 'not stated' otherwise")
    technology_background: str

class Familiarization(BaseModel):
    speakers: list[SpeakerRole]
    profile: ParticipantProfile
    summary: str = Field(description="5-8 sentence summary of the interview")
    initial_observations: list[str] = Field(description="Initial ideas, patterns and notable moments (Braun & Clarke phase 1)")

class UserGroupSummary(BaseModel):
    name: str
    participant_ids: list[str]
    description: str

class DatasetOverview(BaseModel):
    overview: str
    user_groups: list[UserGroupSummary]
    data_quality_notes: list[str]

# ---- Group 2: thematic analysis ----
class Code(BaseModel):
    code: str
    description: str
    quote: str = Field(description="Verbatim supporting quote from the participant")

class CodingResult(BaseModel):
    codes: list[Code]

class Theme(BaseModel):
    theme: str
    definition: str
    sub_themes: list[str]
    codes: list[str]
    participant_ids: list[str]
    groups_present: list[str]
    group_flag: Literal["shared across groups", "group-specific", "divergent across groups"]
    group_differences: str = Field(description="How the user groups differ on this theme, or 'none'")
    key_quotes: list[str]

class ThematicMap(BaseModel):
    themes: list[Theme]
    cross_group_summary: str

# ---- Group 3: theory-informed attributes ----
class AttributeScore(BaseModel):
    attribute: str = Field(description="Attribute key exactly as listed")
    score: Optional[int] = Field(description="1 (very low) to 5 (very high); null if the transcript gives no evidence")
    evidence: str = Field(description="Verbatim quote or precise reason")
    confidence: Literal["low", "medium", "high"]

class LensAssessment(BaseModel):
    scores: list[AttributeScore]
    lens_classification: str = Field(description="The lens-specific verdict, e.g. Rogers adopter category for DOI")
    lens_summary: str

# ---- Group 4: clusters ----
class ClusterInterpretation(BaseModel):
    cluster_id: int
    label: str = Field(description="Short evocative segment name, e.g. 'Cautious Pragmatists'")
    adopter_category: Literal["Innovators", "Early adopters", "Early majority", "Late majority", "Laggards", "Mixed"]
    defining_attributes: list[str]
    explanation: str

class ClusterReport(BaseModel):
    clusters: list[ClusterInterpretation]
    group_vs_cluster_insight: str
    validity_notes: str

# ---- Group 5: persona cards ----
class PersonaCard(BaseModel):
    persona_name: str = Field(description="A realistic first name, not a stereotype")
    archetype: str
    adopter_category: str
    tagline_quote: str = Field(description="A verbatim participant quote that captures the persona")
    bio: str = Field(description="3-4 sentences")
    user_groups: list[str]
    goals: list[str]
    adoption_drivers: list[str]
    adoption_barriers: list[str]
    behaviours: list[str]
    related_themes: list[str]
    design_strategies: list[str] = Field(description="Concrete design and rollout actions that would move this persona toward adoption")
    supporting_quotes: list[str] = Field(description="2-4 verbatim participant quotes")

## Step 5: The theory framework for adoption attributes

You can edit this. Each **lens agent** scores only the attributes in its own lens. Add, remove or rename attributes freely.

In [ ]:
ATTRIBUTES = {
    "Diffusion of Innovation (Rogers)": {
        "innovativeness":      "Openness and willingness to try new technology early; personal innovativeness",
        "relative_advantage":  "Belief that the technology is better than the current way of working",
        "compatibility":       "Fit with existing values, practices, workflows and tools",
        "trialability_need":   "How much they need to try it out at low risk before committing",
        "observability":       "How much seeing others use it, and their results, influences them",
    },
    "TAM / UTAUT": {
        "perceived_usefulness":    "Belief that using it improves their performance (performance expectancy)",
        "perceived_ease_of_use":   "Belief that it is easy to learn and use (effort expectancy)",
        "social_influence":        "Influence of supervisors, peers and the institution on using it",
        "facilitating_conditions": "Availability of infrastructure, access, training and support",
        "self_efficacy":           "Confidence in their own ability to use the technology",
    },
    "UX principles": {
        "trust_in_output":     "Trust in the accuracy and reliability of what the system produces",
        "transparency_need":   "Need to see sources, reasoning and how results are produced",
        "need_for_control":    "Desire to stay in control and verify or override the system",
        "perceived_risk":      "Perceived risks: errors, integrity, privacy, reputation (higher = more risk)",
        "learning_support_need": "Need for onboarding, examples, guidance and help",
    },
}
ATTRIBUTE_KEYS = [a for lens in ATTRIBUTES.values() for a in lens]
LENS_OF = {a: lens for lens, attrs in ATTRIBUTES.items() for a in attrs}
print(f"{len(ATTRIBUTE_KEYS)} attributes across {len(ATTRIBUTES)} lenses")

## Step 6: Agent group 1, familiarization and speaker identification

In [ ]:
READER = f"""You are a qualitative researcher doing the familiarization phase of thematic analysis.
Read the whole interview transcript carefully.
1. Identify every speaker label and decide who is the interviewer and who is the participant
   (interviewers ask questions and steer; participants describe their own experience).
2. Build the participant's profile from what they say. Assign them to one of these user groups:
   {USER_GROUPS or "infer a sensible user group from their role"}.
3. Summarize the interview and note initial observations.
Never invent details that are not in the transcript.
Research context: {RESEARCH_CONTEXT}"""

FAMILIARIZATION_LEAD = """You lead the familiarization phase. You receive the readers' notes for all
interviews. Write an overview of the dataset, confirm the user groups (merge spelling variants,
list the participant IDs in each), and note data-quality issues (very short interviews, unclear
speakers, missing information)."""

def relabel(text: str, speakers, pid: str) -> str:
    """Rewrite speaker labels as INTERVIEWER / PARTICIPANT so later agents never confuse them."""
    for s in sorted(speakers, key=lambda s: -len(s.speaker_label)):
        new = {"interviewer": "INTERVIEWER", "participant": f"PARTICIPANT {pid}"}.get(s.role, "OTHER")
        text = re.sub(rf"(?im)^(\s*(?:\[[^\]]*\]\s*)?){re.escape(s.speaker_label)}(\s*(?:\([^)]*\))?\s*:)",
                      rf"\g<1>{new}\g<2>", text)
    return text

def read_transcript(t):
    result = run_agent(READER, f"Participant ID: {t['participant_id']}\n\nTranscript:\n{t['raw_text']}",
                       Familiarization)
    print(f"  Reader finished {t['participant_id']}")
    return result

print("Group 1: readers are working...")
for t, fam in zip(transcripts, parallel(read_transcript, transcripts)):
    t["familiarization"] = fam
    t["clean_text"] = relabel(t["raw_text"], fam.speakers, t["participant_id"])

overview = run_agent(
    FAMILIARIZATION_LEAD,
    "\n\n".join(f"## {t['participant_id']}\n{t['familiarization'].model_dump_json(indent=1)}" for t in transcripts),
    DatasetOverview,
)
group_of = {pid: g.name for g in overview.user_groups for pid in g.participant_ids}
for t in transcripts:
    t["user_group"] = group_of.get(t["participant_id"], t["familiarization"].profile.user_group)

print("\nUser groups:", {g.name: g.participant_ids for g in overview.user_groups})
for t in transcripts:
    roles = ", ".join(f"{s.speaker_label} = {s.role}" for s in t["familiarization"].speakers)
    print(f"{t['participant_id']} [{t['user_group']}]  {roles}")
save_json("1_familiarization.json", {
    "overview": overview.model_dump(),
    "participants": [{"participant_id": t["participant_id"], "file": t["file"], "user_group": t["user_group"],
                      **t["familiarization"].model_dump(), "clean_transcript": t["clean_text"]} for t in transcripts],
})

## Step 7: Agent group 2, thematic analysis with user groups flagged

In [ ]:
CODER = f"""You are a qualitative coder (Braun & Clarke, phase 2: generating initial codes).
Code ONLY the PARTICIPANT's words, never the interviewer's. Create concise, descriptive codes about
experiences, attitudes, needs, barriers and drivers related to adopting the technology. Every code
needs a short verbatim quote. Research context: {RESEARCH_CONTEXT}"""

THEME_LEAD = f"""You are a senior qualitative researcher (Braun & Clarke, phases 3-5).
Combine the codes from all participants into coherent themes. For each theme, record which
participants and which USER GROUPS it appears in, then FLAG it:
- "shared across groups": appears similarly in every group
- "group-specific": appears only or mainly in one group
- "divergent across groups": groups talk about it in contrasting ways
Explain the group differences. Research context: {RESEARCH_CONTEXT}"""

THEME_REVIEWER = """You review a thematic map (Braun & Clarke, phase 4: reviewing themes).
Check every theme against the codes: is it supported, coherent and distinct from the others?
Are the group flags correct? Merge, split, rename or drop themes where needed, fix wrong flags,
and return the full corrected thematic map."""

def code_transcript(t):
    result = run_agent(CODER, f"User group: {t['user_group']}\n\n{t['clean_text']}", CodingResult)
    print(f"  Coder finished {t['participant_id']}")
    return result

print("Group 2: coders are working...")
for t, coding in zip(transcripts, parallel(code_transcript, transcripts)):
    t["codes"] = coding.codes

all_codes = "\n\n".join(
    f"## {t['participant_id']} (group: {t['user_group']})\n"
    + "\n".join(f"- {c.code}: {c.description} | \"{c.quote}\"" for c in t["codes"])
    for t in transcripts
)
print("Theme lead is building themes...")
draft_map = run_agent(THEME_LEAD, f"User groups: {list(group_of.values())}\n\nCodes:\n{all_codes}",
                      ThematicMap, effort="high")
print("Theme reviewer is checking...")
thematic_map = run_agent(THEME_REVIEWER,
                         f"Codes:\n{all_codes}\n\nDraft thematic map:\n{draft_map.model_dump_json(indent=1)}",
                         ThematicMap, effort="high")

for th in thematic_map.themes:
    print(f"\n■ {th.theme}  [{th.group_flag}]  groups: {', '.join(th.groups_present)}")
    print(f"  {th.definition}")
    if th.group_differences.lower() != "none":
        print(f"  Group differences: {th.group_differences}")
save_json("2_thematic_analysis.json", {
    "codes": {t["participant_id"]: [c.model_dump() for c in t["codes"]] for t in transcripts},
    "thematic_map": thematic_map.model_dump(),
})

## Step 8: Agent group 3, theory lens agents score the adoption attributes

In [ ]:
import pandas as pd

theme_brief = "\n".join(f"- {th.theme} [{th.group_flag}]: {th.definition}" for th in thematic_map.themes)

def lens_prompt(lens: str) -> str:
    attrs = "\n".join(f"- {k}: {v}" for k, v in ATTRIBUTES[lens].items())
    return f"""You are an expert in {lens} applied to technology adoption research.
Assess ONE participant through the {lens} lens only. Score each attribute below from 1 (very low)
to 5 (very high), based strictly on what the PARTICIPANT says. If there is no evidence, return
score null; do not guess. Give a verbatim quote or precise reason as evidence.
Attributes:
{attrs}
In lens_classification give your lens-specific verdict (for Diffusion of Innovation: the likely
Rogers adopter category; for TAM/UTAUT: likely behavioural intention to use; for UX: the participant's
top UX priority). Research context: {RESEARCH_CONTEXT}"""

def assess(job):
    t, lens = job
    task = (f"Participant {t['participant_id']} (user group: {t['user_group']})\n"
            f"Profile: {t['familiarization'].profile.model_dump_json()}\n\n"
            f"Themes found in the study:\n{theme_brief}\n\nTranscript:\n{t['clean_text']}")
    result = run_agent(lens_prompt(lens), task, LensAssessment)
    print(f"  {lens} agent finished {t['participant_id']}")
    return result

jobs = [(t, lens) for t in transcripts for lens in ATTRIBUTES]
print(f"Group 3: {len(jobs)} lens assessments ({len(ATTRIBUTES)} lenses x {len(transcripts)} participants)...")
rows, evidence = [], []
for t in transcripts:
    t["lenses"] = {}
for (t, lens), res in zip(jobs, parallel(assess, jobs)):
    t["lenses"][lens] = res
    for s in res.scores:
        if s.attribute in ATTRIBUTE_KEYS:
            evidence.append({"participant_id": t["participant_id"], "user_group": t["user_group"], "lens": lens,
                             "attribute": s.attribute, "score": s.score, "confidence": s.confidence,
                             "evidence": s.evidence})

evidence_df = pd.DataFrame(evidence)
scores = (evidence_df.pivot_table(index="participant_id", columns="attribute", values="score", aggfunc="mean")
          .reindex(columns=ATTRIBUTE_KEYS))
scores.insert(0, "user_group", [group_of.get(p, "") or next(t["user_group"] for t in transcripts
                                if t["participant_id"] == p) for p in scores.index])
group_profile = scores.groupby("user_group")[ATTRIBUTE_KEYS].mean().round(2)

ATTRIBUTE_SYNTHESIZER = f"""You are a technology-adoption researcher. Using Diffusion of Innovation,
TAM/UTAUT and UX principles, describe the adoption attribute profile of EACH user group: which
attributes are strong or weak, how the groups differ, and what that means for adoption. Cite the
scores and themes. Write concise Markdown. Research context: {RESEARCH_CONTEXT}"""
print("Attribute synthesizer is comparing user groups...")
group_attribute_report = run_agent(
    ATTRIBUTE_SYNTHESIZER,
    f"Mean attribute scores by user group (1-5):\n{group_profile.to_string()}\n\n"
    f"Participant scores:\n{scores.to_string()}\n\nThemes:\n{theme_brief}",
    effort="high",
)

evidence_df.to_csv(os.path.join(OUTPUT_DIR, "3_attribute_evidence.csv"), index=False)
scores.to_csv(os.path.join(OUTPUT_DIR, "3_attribute_scores.csv"))
from IPython.display import Markdown, display
display(scores.style.format(precision=1, na_rep="–").background_gradient(cmap="Blues", vmin=1, vmax=5,
                                                                         subset=ATTRIBUTE_KEYS))
display(Markdown(group_attribute_report))

## Step 9: Agent group 4, cluster analysis

The clustering itself is statistical, done in Python. Participants are grouped by their attribute scores, not by their user group, so a segment can mix Faculty and PhD students. The **Cluster Analyst** agent then interprets each cluster through Rogers' adopter categories.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram

FORCE_K = None     # set e.g. 3 to choose the number of clusters yourself
MAX_K = 6

X = scores[ATTRIBUTE_KEYS].copy()
X = X.fillna(X.median()).fillna(3)          # missing evidence -> median of other participants
Xs = StandardScaler().fit_transform(X)
n = len(X)

sil = {}
for k in range(2, min(MAX_K, n - 1) + 1):
    labels_k = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(Xs)
    sil[k] = silhouette_score(Xs, labels_k)
k = FORCE_K or (max(sil, key=sil.get) if sil else 1)
labels = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(Xs) if k > 1 else np.zeros(n, int)
scores["cluster"] = labels + 1
print("Silhouette by k:", {kk: round(v, 3) for kk, v in sil.items()}, f"-> using k = {k}")
print(pd.crosstab(scores["cluster"], scores["user_group"]))

centroids = scores.groupby("cluster")[ATTRIBUTE_KEYS].mean().round(2)

# --- charts ---
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
MARKERS = ["o", "s", "^", "D", "P", "X"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dendrogram(linkage(Xs, "ward"), labels=list(X.index), ax=axes[0], color_threshold=0,
           above_threshold_color="#52514e")
axes[0].set_title("Hierarchical clustering (Ward)", loc="left")
axes[0].spines[["top", "right"]].set_visible(False)

pts = PCA(n_components=2).fit_transform(Xs) if n > 2 else np.c_[Xs[:, 0], np.zeros(n)]
groups = sorted(scores["user_group"].unique())
for c in sorted(scores["cluster"].unique()):
    for gi, g in enumerate(groups):
        m = (scores["cluster"].values == c) & (scores["user_group"].values == g)
        if m.any():
            axes[1].scatter(pts[m, 0], pts[m, 1], s=90, color=SERIES[(c - 1) % 8], marker=MARKERS[gi % 6],
                            edgecolor="white", linewidth=2, label=f"Cluster {c} · {g}")
for (x, y), pid in zip(pts, X.index):
    axes[1].annotate(pid, (x, y), xytext=(6, 4), textcoords="offset points", fontsize=9, color="#52514e")
axes[1].set_title("Participants in attribute space (PCA)", loc="left")
axes[1].legend(frameon=False, fontsize=9)
axes[1].spines[["top", "right"]].set_visible(False)
axes[1].grid(alpha=0.2)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "4_clusters.png"), dpi=150); plt.show()

display(centroids.style.format(precision=1).background_gradient(cmap="Blues", vmin=1, vmax=5))

CLUSTER_ANALYST = f"""You are a technology-adoption researcher interpreting a cluster analysis.
For each cluster, use the centroid scores (1-5), its members and their user groups to name the segment,
map it to a Rogers adopter category, and explain the defining attributes using Diffusion of Innovation,
TAM/UTAUT and UX principles. Comment on how clusters relate to the user groups, and be honest about
validity (sample size, overlap). Research context: {RESEARCH_CONTEXT}"""

members = {c: list(scores.index[scores["cluster"] == c]) for c in centroids.index}
print("Cluster analyst is interpreting...")
cluster_report = run_agent(
    CLUSTER_ANALYST,
    f"Centroids:\n{centroids.to_string()}\n\nMembers and groups:\n"
    + "\n".join(f"Cluster {c}: " + ", ".join(f"{p} ({scores.loc[p, 'user_group']})" for p in ps)
                for c, ps in members.items())
    + f"\n\nSilhouette by k: {sil}\n\nThemes:\n{theme_brief}\n\nGroup attribute report:\n{group_attribute_report}",
    ClusterReport, effort="high",
)
for ci in cluster_report.clusters:
    print(f"\nCluster {ci.cluster_id}: {ci.label}  ({ci.adopter_category})")
    print("  " + ci.explanation)
print("\nUser groups vs clusters:", cluster_report.group_vs_cluster_insight)
print("Validity:", cluster_report.validity_notes)
save_json("4_clusters.json", {"k": int(k), "silhouette": sil, "members": members,
                              "centroids": centroids.to_dict(orient="index"), "report": cluster_report.model_dump()})

## Step 10: Agent group 5, persona cards

In [ ]:
PERSONA_DESIGNER = f"""You are a UX researcher who designs evidence-based ADOPTION personas.
Write one persona for the given cluster. Base every element on the cluster's attribute scores,
the members' transcripts and the themes. Quotes MUST be copied verbatim from the participants
(never the interviewer). Avoid stereotypes. Design strategies must be concrete actions that move this
persona toward adoption (onboarding, features, communication, institutional support).
Research context: {RESEARCH_CONTEXT}"""

interp = {ci.cluster_id: ci for ci in cluster_report.clusters}
by_pid = {t["participant_id"]: t for t in transcripts}

def design_persona(c):
    ci = interp.get(c)
    task = (f"Cluster {c}: {ci.label if ci else ''} ({ci.adopter_category if ci else ''})\n"
            f"{ci.explanation if ci else ''}\n\n"
            f"Centroid scores (1-5):\n{centroids.loc[c].to_string()}\n\n"
            f"Themes:\n{theme_brief}\n\nMember transcripts:\n\n"
            + "\n\n".join(f"### {p} ({by_pid[p]['user_group']})\n{by_pid[p]['clean_text']}" for p in members[c]))
    card = run_agent(PERSONA_DESIGNER, task, PersonaCard, effort="high")
    print(f"  Persona designer finished cluster {c}")
    return card

print("Group 5: persona designers are working...")
cards = dict(zip(centroids.index, parallel(design_persona, list(centroids.index))))

# Check that every quote really appears in that cluster's transcripts
def _norm(s): return re.sub(r"[^a-z0-9 ]", "", re.sub(r"\s+", " ", s.lower())).strip()
def verified(quote, c):
    q = _norm(quote.strip('"“”'))
    return bool(q) and any(q in _norm(by_pid[p]["raw_text"]) for p in members[c])

In [ ]:
import html as _html

def _li(items): return "".join(f"<li>{_html.escape(i)}</li>" for i in items)
def _q(quote, c):
    mark = "✓" if verified(quote, c) else "⚠ not found in transcripts"
    return f'<blockquote>“{_html.escape(quote.strip(chr(34)))}”<span class="chk">{mark}</span></blockquote>'

def card_html(c, card):
    color = SERIES[(c - 1) % 8]
    bars = ""
    for lens, attrs in ATTRIBUTES.items():
        bars += f'<div class="lens">{_html.escape(lens)}</div>'
        for a in attrs:
            v = centroids.loc[c, a]
            width, label = (0, "–") if pd.isna(v) else (v / 5 * 100, f"{v:.1f}")
            bars += (f'<div class="bar"><span>{a.replace("_", " ")}</span>'
                     f'<div class="track"><div class="fill" style="width:{width:.0f}%"></div></div>'
                     f'<b>{label}</b></div>')
    initials = "".join(w[0] for w in card.persona_name.split()[:2]).upper()
    return f"""
<div class="card">
  <div class="head" style="border-top:6px solid {color}">
    <div class="avatar" style="background:{color}">{_html.escape(initials)}</div>
    <div><h2>{_html.escape(card.persona_name)}</h2><div class="arch">{_html.escape(card.archetype)}</div>
    <span class="badge">{_html.escape(card.adopter_category)}</span>
    <span class="meta">Cluster {c} · {len(members[c])} participants · {_html.escape(", ".join(card.user_groups))}</span></div>
  </div>
  {_q(card.tagline_quote, c)}
  <p>{_html.escape(card.bio)}</p>
  <h3>Adoption attributes <small>(cluster mean, 1–5; – = no evidence)</small></h3>{bars}
  <div class="cols">
    <div><h3>Goals</h3><ul>{_li(card.goals)}</ul></div>
    <div><h3>Behaviours</h3><ul>{_li(card.behaviours)}</ul></div>
    <div><h3>Adoption drivers</h3><ul>{_li(card.adoption_drivers)}</ul></div>
    <div><h3>Adoption barriers</h3><ul>{_li(card.adoption_barriers)}</ul></div>
  </div>
  <h3>Design &amp; rollout strategies</h3><ul>{_li(card.design_strategies)}</ul>
  <h3>Related themes</h3><div>{"".join(f'<span class="tag">{_html.escape(t)}</span>' for t in card.related_themes)}</div>
  <h3>In their words</h3>{"".join(_q(q, c) for q in card.supporting_quotes)}
</div>"""

CSS = """
:root{--bg:#f4f4f2;--card:#fcfcfb;--ink:#0b0b0b;--ink2:#52514e;--line:#e4e3df;--track:#e9e8e4;--fill:#2a78d6}
@media (prefers-color-scheme: dark){:root{--bg:#111110;--card:#1a1a19;--ink:#f0efec;--ink2:#c3c2b7;--line:#383835;--track:#2c2c2a;--fill:#3987e5}}
body{background:var(--bg);margin:0}
.wrap{font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;color:var(--ink);display:grid;
      grid-template-columns:repeat(auto-fill,minmax(380px,1fr));gap:20px;padding:16px;background:var(--bg)}
.card{background:var(--card);border:1px solid var(--line);border-radius:14px;padding:20px;font-size:14px;line-height:1.45}
.head{display:flex;gap:14px;align-items:center;margin:-20px -20px 8px;padding:16px 20px 0;border-radius:14px 14px 0 0}
.avatar{width:56px;height:56px;border-radius:50%;color:#fff;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:20px;flex:none}
h2{margin:0;font-size:20px} h3{font-size:12px;text-transform:uppercase;letter-spacing:.06em;color:var(--ink2);margin:16px 0 6px}
.arch{color:var(--ink2)} .badge{display:inline-block;margin-top:4px;padding:2px 8px;border-radius:99px;border:1px solid var(--line);font-size:12px;font-weight:600}
.meta{display:block;font-size:12px;color:var(--ink2);margin-top:4px}
blockquote{margin:10px 0;padding:8px 12px;border-left:3px solid var(--line);font-style:italic}
.chk{display:block;font-style:normal;font-size:11px;color:var(--ink2);margin-top:2px}
.lens{font-size:12px;font-weight:600;margin:8px 0 2px}
.bar{display:grid;grid-template-columns:150px 1fr 30px;gap:8px;align-items:center;font-size:12px;color:var(--ink2)}
.track{height:8px;background:var(--track);border-radius:4px}.fill{height:8px;background:var(--fill);border-radius:4px}
.bar b{color:var(--ink);text-align:right}
.cols{display:grid;grid-template-columns:1fr 1fr;gap:0 16px} ul{margin:0;padding-left:18px}
.tag{display:inline-block;margin:0 6px 6px 0;padding:2px 8px;border-radius:6px;background:var(--track);font-size:12px}
@media (max-width:480px){.cols{grid-template-columns:1fr}.wrap{grid-template-columns:1fr}}
"""

cards_html = f'<style>{CSS}</style><div class="wrap">' + "".join(card_html(c, cards[c]) for c in cards) + "</div>"
from IPython.display import HTML
display(HTML(cards_html))

page = f'<!doctype html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Adoption Personas</title></head><body>{cards_html}</body></html>'
with open(os.path.join(OUTPUT_DIR, "5_persona_cards.html"), "w", encoding="utf-8") as f:
    f.write(page)
save_json("5_persona_cards.json", {int(c): card.model_dump() for c, card in cards.items()})

## Step 11: Download everything

In [ ]:
import shutil
shutil.make_archive("persona_outputs", "zip", OUTPUT_DIR)
print("Files:", sorted(os.listdir(OUTPUT_DIR)))
try:
    from google.colab import files
    files.download("persona_outputs.zip")
except ImportError:
    print("Saved persona_outputs.zip")